# Amiri pipeline -- other real-life trims (peak_0.6/0.7/0.8, magnitude_1/2/3)



## Real

In [ ]:
import sys, warnings, math, json, pickle, time, shutil
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import pm4py
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from hyperopt import tpe, Trials, hp, fmin, STATUS_OK

ROOT = Path('.').resolve().parent.parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "plain-field"))

from time_series_preprocessing import ts_splits_from_log
from create_prefixes_from_windows import load_event_log, make_three_way_split
from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
from setttings import set_global_seed
from hpo_val_scoring import build_val_as_test_df
from amiri.trainer import AmiriTrainer, select_amiri_hpo_winner
from amiri.params  import default_params
from amiri.converter import convert
from pf_lukas_prediction import REAL_LOGS, REAL_TRIMS

RESULTS     = ROOT / 'results'
BEST_MODELS = ROOT / 'best_models'

_COLS = {
    'case:concept:name':    'caseid',
    'concept:name':         'task',
    'lifecycle:transition': 'event_type',
    'org:resource':         'user',
    'time:timestamp':       'end_timestamp',
}

def _trim_kw(trim_str):
    base = dict(trim_pct=0.25, trim_k=1.5, trim_frac=0.6, trim_window=7)
    if trim_str == 'none':
        return {**base, 'trim_method': None}
    method, val = trim_str.rsplit('_', 1)
    val = float(val)
    if method == 'magnitude': return {**base, 'trim_method': 'magnitude', 'trim_k': val}
    if method == 'pct':       return {**base, 'trim_method': 'pct',       'trim_pct': val}
    if method == 'peak':      return {**base, 'trim_method': 'peak',      'trim_frac': val}
    raise ValueError(f'Unknown trim: {trim_str}')

def rem_time_to_event_log(rem_df):
    """Build a 2-event-per-case log: case start + predicted end."""
    rt = rem_df.copy()
    rt['start_timestamp']  = pd.to_datetime(rt['start_timestamp'])
    rt['anchor_timestamp'] = pd.to_datetime(rt['anchor_timestamp'])
    rt['predicted_end']    = rt['anchor_timestamp'] + pd.to_timedelta(rt['rem_time_days'], unit='D')
    return pd.concat([
        rt[['caseid', 'start_timestamp']].rename(columns={'start_timestamp': 'end_timestamp'}),
        rt[['caseid', 'predicted_end']].rename(columns={'predicted_end': 'end_timestamp'}),
    ], ignore_index=True)

def pt_kpi_series(event_log, test_index_cc, test_index_tt, window='days'):
    pred_cc = create_concurrent_cases_timeseries(
        event_log, time_col='end_timestamp', case_col='caseid', window=window, plot=False)
    pred_tt = create_avg_throughtput_time_timeseries(
        event_log, time_col='end_timestamp', case_col='caseid', window=window, plot=False)
    cc_arr = pred_cc.reindex(test_index_cc).ffill().bfill().fillna(0).to_numpy()
    tt_arr = pred_tt.reindex(test_index_tt).ffill().bfill().fillna(0).to_numpy()
    return cc_arr, tt_arr

def _save_metrics_and_plot(out_dir, run_name, model_tag, cc, tt, cc_pred, tt_pred):
    out_dir.mkdir(parents=True, exist_ok=True)
    cc_actual = cc['test'].to_numpy()
    tt_actual = tt['test'].to_numpy()
    results = {
        'concurrent_cases': {'mse': mean_squared_error(cc_actual, cc_pred),
                             'mae': mean_absolute_error(cc_actual, cc_pred)},
        'throughput_time':  {'mse': mean_squared_error(tt_actual, tt_pred),
                             'mae': mean_absolute_error(tt_actual, tt_pred)},
    }
    pd.DataFrame([
        dict(dataset=run_name, series=s, model=model_tag, mse=m['mse'], mae=m['mae'])
        for s, m in results.items()
    ]).to_csv(out_dir / f'metrics_{run_name}.csv', index=False)
    for series_name, actual, pred, index in [
        ('concurrent_cases', cc_actual, cc_pred, cc['test'].index),
        ('throughput_time',  tt_actual, tt_pred, tt['test'].index),
    ]:
        m = results[series_name]
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.plot(index, actual, color='green',  label='actual',          linewidth=1.5)
        ax.plot(index, pred,   color='purple', label=f'{model_tag} (pred)', linestyle='--')
        ax.set_title(f'{run_name} — {series_name}  MSE={m["mse"]:.4f}  MAE={m["mae"]:.4f}')
        ax.legend(); plt.tight_layout()
        plt.savefig(out_dir / f'{series_name}.png', dpi=150, bbox_inches='tight')
        plt.show(); plt.close()

set_global_seed(1904)
print(f'{len(REAL_LOGS)} real-life datasets x {len(REAL_TRIMS)} non-ssd trims')


### 1. First (full-trace)

In [ ]:
AMIRI_MAX_EVALS = 12
AMIRI_SPACE = {
    'gt_layers':        hp.choice('gt_layers',    [3, 5, 7]),
    'gt_n_heads':       hp.choice('gt_n_heads',   [4, 8]),
    'gt_dim_hidden':    hp.choice('gt_dim_hidden', [32, 64, 128]),
    'gt_dropout':       hp.uniform('gt_dropout',  0.0, 0.4),
    'gt_attn_dropout':  hp.uniform('gt_attn_dropout', 0.0, 0.6),
    'base_lr':          hp.loguniform('base_lr',  np.log(1e-4), np.log(1e-2)),
    'weight_decay':     hp.loguniform('weight_decay', np.log(1e-4), np.log(1e-1)),
    'batch_size':       hp.choice('batch_size',   [64, 128, 256]),
    'max_epoch':        hp.choice('max_epoch',    [50, 100]),
}

for name in REAL_LOGS:
    for trim in REAL_TRIMS:
        RUN_NAME = f'{name}_test_full'

        metrics_path = RESULTS / 'amiri_hpo' / trim / RUN_NAME / f'metrics_{RUN_NAME}.csv'
        if metrics_path.exists():
            print(f'[skip] {name}/{trim}: metrics already exist')
            continue

        print(f"\n{'='*60}\n{name} / {trim} (amiri HPO + first)\n{'='*60}")

        log = pm4py.read_xes(str(ROOT / 'data' / 'real-life' / f'{name}.xes'))
        ts = ts_splits_from_log(log, **_trim_kw(trim), train_frac=0.7, val_frac=0.1, cut_date=None)
        cc = ts['concurrent_cases']
        tt = ts['throughput_time']

        train_split = cc['train_split']
        val_split   = cc['val_split']

        df_raw = load_event_log(ROOT / 'data' / 'real-life' / f'{name}.xes',
                                time_col='time:timestamp', case_col='case:concept:name')
        df = df_raw.rename(columns=_COLS)
        df['task'] = df['task'].fillna('unk')
        df['user'] = df['user'].fillna('unk') if 'user' in df.columns else 'unk'

        train_df, val_df, test_df = make_three_way_split(
            df, case_col='caseid', time_col='end_timestamp',
            train_split=train_split, val_split=val_split, full_traces=True,
        )

        val_as_test_df = build_val_as_test_df(df, train_split)

        hpo_dir = BEST_MODELS / name / trim / 'amiri' / 'hpo_trials'
        hpo_dir.mkdir(parents=True, exist_ok=True)
        shared_dataset_dir = hpo_dir / 'shared_dataset'
        shared_dataset_dir.mkdir(parents=True, exist_ok=True)

        _trials_pkl = hpo_dir / 'hyperopt_trials.pkl'
        if _trials_pkl.exists():
            with open(_trials_pkl, 'rb') as _f:
                _hpo_trials = pickle.load(_f)
            _completed = len(_hpo_trials.trials)
        else:
            _hpo_trials = Trials()
            _completed  = 0

        _raw_meta = shared_dataset_dir / 'AMIRI' / 'raw' / 'meta.pkl'
        if not _raw_meta.exists():
            convert(train_df, val_df, test_df, shared_dataset_dir, seed=42)

        amiri_hpo_results = []
        _trial_counter    = [_completed]

        def _amiri_objective(trial_cfg):
            i = _trial_counter[0]
            _trial_counter[0] += 1
            trial_id  = f'trial_{i:03d}'
            trial_dir = hpo_dir / trial_id
            trial_dir.mkdir(parents=True, exist_ok=True)

            with open(_trials_pkl, 'wb') as _f:
                pickle.dump(_hpo_trials, _f)

            cfg_path = trial_dir / 'config.json'
            if not cfg_path.exists():
                cfg_path.write_text(json.dumps(trial_cfg, indent=2))

            result_path = trial_dir / 'result.json'
            if result_path.exists():
                res = json.loads(result_path.read_text())
                amiri_hpo_results.append(res)
                return {'loss': res['val_mae'], 'status': STATUS_OK, **res}

            params = default_params(**trial_cfg)
            params['seed'] = 42
            trainer = AmiriTrainer(
                train_df, val_df, test_df,
                run_name=trial_id, params=params,
                output_dir=trial_dir, dataset_dir=shared_dataset_dir,
            )
            try:
                trainer.run()
                val_mae = trainer.get_best_val_mae()
            except Exception as e:
                print(f'[{trial_id}] FAILED: {e}')
                res = {'trial_id': trial_id, 'val_mae': float('inf'), 'error': str(e), **trial_cfg}
                result_path.write_text(json.dumps(res, indent=2))
                amiri_hpo_results.append(res)
                return {'loss': float('inf'), 'status': STATUS_OK, **res}

            res = {'trial_id': trial_id, 'val_mae': val_mae, **trial_cfg}
            result_path.write_text(json.dumps(res, indent=2))
            amiri_hpo_results.append(res)
            print(f'[{trial_id}] val_mae={val_mae:.4f}')
            return {'loss': val_mae, 'status': STATUS_OK, **res}

        best_amiri = fmin(
            fn=_amiri_objective, space=AMIRI_SPACE, algo=tpe.suggest,
            max_evals=AMIRI_MAX_EVALS, trials=_hpo_trials, verbose=False,
        )
        with open(_trials_pkl, 'wb') as _f:
            pickle.dump(_hpo_trials, _f)

        if not amiri_hpo_results:
            for _rp in sorted(hpo_dir.glob('trial_*/result.json')):
                try:
                    amiri_hpo_results.append(json.loads(_rp.read_text()))
                except Exception:
                    pass

        winner = select_amiri_hpo_winner(
            hpo_dir, train_df, val_df, val_as_test_df, cc['val'], tt['val'],
            default_params_fn=default_params,
        )
        if winner is None:
            print(f'  [SKIP] {name}/{trim}: val-CC-MAE rescoring found no usable trial')
            continue

        best_model_dir = BEST_MODELS / name / trim / 'amiri' / RUN_NAME
        best_model_dir.mkdir(parents=True, exist_ok=True)

        _gps_src = winner['gps_results_dir']
        _gps_dst = best_model_dir / 'gps_results'
        if _gps_src.exists() and not _gps_dst.exists():
            shutil.copytree(_gps_src, _gps_dst)

        _best_params = default_params(**{
            k: v.item() if hasattr(v, 'item') else v
            for k, v in winner['cfg'].items()
        })
        _best_params['seed'] = 42
        _best_params['selection_metric'] = 'val_cc_mae'
        _best_params['val_cc_mae'] = winner['val_cc_mae']
        _best_params['val_tt_mae'] = winner['val_tt_mae']
        (best_model_dir / 'best_params.json').write_text(json.dumps(_best_params, indent=2))
        print(f'  Winning trial: {winner["trial_id"]}  val_cc_mae={winner["val_cc_mae"]:.4f}')

        final_trainer = AmiriTrainer(
            train_df, val_df, test_df, run_name=RUN_NAME,
            params=_best_params, output_dir=best_model_dir, dataset_dir=shared_dataset_dir,
        )
        _ckpt_base = best_model_dir / 'gps_results' / 'amiri_gps'
        if not _ckpt_base.exists():
            final_trainer.run()

        rem_time_df = final_trainer.predict()
        event_log_a = rem_time_to_event_log(rem_time_df)
        cc_pred, tt_pred = pt_kpi_series(event_log_a, cc['test'].index, tt['test'].index)
        _save_metrics_and_plot(metrics_path.parent, RUN_NAME, 'amiri_hpo', cc, tt, cc_pred, tt_pred)
        print(f'  saved -> {metrics_path.parent}')


### 2. Half-prefix

In [ ]:
for name in REAL_LOGS:
    for trim in REAL_TRIMS:
        RUN_NAME = f'{name}_test_full'

        best_model_dir = BEST_MODELS / name / trim / 'amiri' / RUN_NAME
        best_params_path = best_model_dir / 'best_params.json'
        if not best_params_path.exists():
            print(f'[skip] {name}/{trim}: no trained model')
            continue

        metrics_path = RESULTS / 'amiri_hpo_half' / trim / RUN_NAME / f'metrics_{RUN_NAME}.csv'
        if metrics_path.exists():
            print(f'[skip] {name}/{trim}: half-prefix metrics already exist')
            continue

        print(f"\n{'='*60}\n{name} / {trim} (amiri half-prefix)\n{'='*60}")

        log = pm4py.read_xes(str(ROOT / 'data' / 'real-life' / f'{name}.xes'))
        ts = ts_splits_from_log(log, **_trim_kw(trim), train_frac=0.7, val_frac=0.1, cut_date=None)
        cc = ts['concurrent_cases']
        tt = ts['throughput_time']

        df_raw = load_event_log(ROOT / 'data' / 'real-life' / f'{name}.xes',
                                time_col='time:timestamp', case_col='case:concept:name')
        df = df_raw.rename(columns=_COLS)
        df['task'] = df['task'].fillna('unk')
        df['user'] = df['user'].fillna('unk') if 'user' in df.columns else 'unk'

        train_split = cc['train_split']
        val_split   = cc['val_split']
        train_df, val_df, test_df_std = make_three_way_split(
            df, case_col='caseid', time_col='end_timestamp',
            train_split=train_split, val_split=val_split, full_traces=True,
        )

        _test_cids    = set(test_df_std['caseid'].astype(str))
        _df_test_full = (df[df['caseid'].astype(str).isin(_test_cids)]
                         .copy().sort_values(['caseid', 'end_timestamp']))
        _half_parts = []
        for _cid, _grp in _df_test_full.groupby('caseid', sort=False):
            _half_parts.append(_grp.iloc[: max(1, math.ceil(len(_grp) / 2))])
        test_df = pd.concat(_half_parts, ignore_index=True)

        half_dir            = best_model_dir / 'half_prefix'
        half_dataset_dir    = half_dir / 'dataset'
        shared_dataset_dir  = best_model_dir.parent / 'hpo_trials' / 'shared_dataset'
        half_dir.mkdir(parents=True, exist_ok=True)

        _gps_link = half_dir / 'gps_results'
        if not _gps_link.exists():
            _gps_link.symlink_to(best_model_dir / 'gps_results')

        best_params = json.loads(best_params_path.read_text())
        trainer_a = AmiriTrainer(
            train_df, val_df, test_df,
            run_name=name, params=best_params,
            output_dir=half_dir, dataset_dir=half_dataset_dir,
            ref_dataset_dir=shared_dataset_dir,
        )
        rem_time_df = trainer_a.predict()
        event_log_a = rem_time_to_event_log(rem_time_df)
        cc_pred, tt_pred = pt_kpi_series(event_log_a, cc['test'].index, tt['test'].index)
        _save_metrics_and_plot(metrics_path.parent, RUN_NAME, 'amiri_hpo_half', cc, tt, cc_pred, tt_pred)
        print(f'  saved -> {metrics_path.parent}')


### 3. Plain-field

In [ ]:
from pf_lukas_prediction import run_pf_job

for ds in REAL_LOGS:
    for trim in REAL_TRIMS:
        run_pf_job(ds, trim, is_real=True, do_bukhsh=False, do_camargo=False)
